In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import multiprocessing as mp

# 1. Path Resolution
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/pavel/academia/wintermute/mf-temporary')

# 2. The Great MPI Blindfold
keys_to_delete = [k for k in os.environ if k.startswith('SLURM') or k.startswith('OMPI_') or k.startswith('PRTE_') or 'HYDRA' in k]
for key in keys_to_delete:
    del os.environ[key]

# Prevent CPU hogging by the C++ backend
os.environ["OMP_NUM_THREADS"] = "1" 

# 3. Safe Multiprocessing
try:
    mp.set_start_method('spawn', force=True)
    print("Multiprocessing context set to:", mp.get_start_method())
except RuntimeError:
    pass

print(f"Loaded paths securely. MPI blindfolded. Ready for testing!")

In [ ]:
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from codes.neuron_simulation.config import NeuronSimulationConfig
from codes.network_params.loader import load_network_parameters
from codes.plotting import fig_plots
import codes.neuron_simulation as ns

import codes.stimuli as stim

from codes.snn_simulation.config import SpikingNeuralNetworkSimulationConfig

from pydantic import BaseModel

project_path = repo_path / "projects" / "00_drafts_tests"
os.chdir(project_path)

class TestNeuronSimulationParams(BaseModel):
    neuron_simulation: NeuronSimulationConfig

class TestSNNSimulationParams(BaseModel):
    snn_simulation: SpikingNeuralNetworkSimulationConfig


In [ ]:
network_params = load_network_parameters(project_path / "params" / "network_params_new.yaml")
simulators = ["zerlaut2018", "pynn.nest"]


In [ ]:
from codes.stimuli.loader import load_stimuli_config
stimuli_config = load_stimuli_config(project_path / "params" / "stimuli.yaml")

In [ ]:
test_workflow_params = {}
test_workflow_params["snn_simulation"] = {
    "network_name": "divolo_static",
    "execution_mode": "run",
    "simulator": "pynn.nest",
    "time_step": 0.1,
    "seed": 42,
    "n_runs": 5,
    "smoothing": {
        "function": "sliding_window",
        "time_constant": 50,
        "kwargs": {}
    },
    "init_values": {
        "exc_neuron": {
            "voltage": -65.0,
            "adaptation": 0.0
        },
        "inh_neuron": {
            "voltage": -65.0,
            "adaptation": 0.0
        }

    },
    "recorded_samples": {
        "exc_neuron": 500,
        "inh_neuron": 500
    }
}

snn_sim_params = TestSNNSimulationParams(**test_workflow_params).snn_simulation

In [ ]:
from codes.snn_simulation import run_snn_simulation_workflow


snn_results = run_snn_simulation_workflow(
        snn_sim_params=snn_sim_params, 
        network_params=network_params,
        stimuli_dict=stimuli_config
    )

In [ ]:
reload(fig_plots)


In [ ]:
for snn_result in snn_results.values():
    snn_result.DEFAULT_UNITS["stim_rate_mean"] = "Hz"

In [ ]:
for stimulus_name in stimuli_config.keys():
    fig_plots.fig_full_network_overview_together(
        snn_results[stimulus_name], 
        [],
        common_params={
            'xmargin': 0.0,
            'ymargin': 0.0,
            'labels': ['SNN'] + [],
            'legend': {'loc': 'upper left'},
            'xlim' : (0, snn_results[stimulus_name].times()[-1])
        },
        fig_params={
            'figsize': (20, 10),  # width, height
            'tight_layout': True,
            'savefig': True,
            'savefig_path': project_path / "imgs" / f"{stimulus_name}_Full_network_overview_together.png",
            'title' : f"Network overview for stimulus: '{stimulus_name}' and drive rate: {stimuli_config[stimulus_name].drive_rate} Hz",
        }
    )

    display(Image(filename=str(project_path / "imgs" / f"{stimulus_name}_Full_network_overview_together.png")))
